# IoMT Ransomware Detection
### Project Overview & Progress

**Goal:** Detect ransomware attacks on medical devices early enough to stop them before they cause harm.

---

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.gridspec as gridspec
from IPython.display import display, Markdown
from torch.utils.data import DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams["figure.dpi"] = 120

if os.path.basename(os.getcwd()) != "IoMT_ransom":
    os.chdir("IoMT_ransom")

from src.config import Config
from src.utils import set_seed, get_device
from src.models.autoencoder import Autoencoder
from src.models.mamba_classifier import MambaClassifier
from src.models.lstm_classifier import LSTMClassifier
from src.datasets.sequence_dataset import ArraySequenceDataset
from src.evaluate.metrics import compute_metrics

cfg    = Config()
device = get_device()
set_seed(cfg.random_seed)
print(f"Device : {device}")
print(f"CWD    : {os.getcwd()}")

---
## 1. The Problem

Medical devices — ventilators, infusion pumps, patient monitors — are increasingly connected to hospital networks. That connectivity makes them vulnerable to ransomware: malware that quietly encrypts a device's files until a ransom is paid. In a clinical setting, a locked device isn't just an IT problem — it can directly affect patient care.

**The challenge is timing.** Ransomware doesn't announce itself. It starts slowly: a slight increase in disk activity, unusual network traffic, small changes in device behaviour. By the time it's obvious something is wrong, the damage may already be done.

**Our goal is early detection** — catching an attack in its first few timesteps, while there is still time to isolate the device and prevent it from spreading across the network.

### Our Approach

We built a two-stage detection pipeline:

**Stage 1 — Learn what normal looks like.**
We train an Autoencoder on healthy device data only. The autoencoder compresses each timestep of sensor data into a compact summary, and then reconstructs it. When a device behaves normally, reconstruction is accurate. When something unusual is happening, reconstruction error spikes — giving us an anomaly signal without ever needing labelled attack data to train on.

**Stage 2 — Classify sequences as attack or benign.**
Instead of looking at one timestep at a time, we pass a sliding window of 20 timesteps through a Mamba classifier. The model reads the sequence in order and learns the *pattern* of an unfolding attack — not just whether a single moment looks unusual, but whether the last 20 timesteps tell the story of ransomware taking hold.

---
## 2. Datasets

We trained and tested on two datasets, with a third currently being added.

In [ ]:
sim_csv = os.path.join(cfg.raw_icu_path, "icu_simulation.csv")
df_sim  = pd.read_csv(sim_csv)
df_sim["status"] = df_sim["device_id"].apply(lambda x: "Attack" if "attack" in x else "Benign")

X_ton = np.load(os.path.join(cfg.ton_splits_path, "X_test.npy"))
y_ton = np.load(os.path.join(cfg.ton_splits_path, "y_test.npy"))

n_dev = df_sim["device_id"].nunique()
print("=== Simulated ICU Dataset ===")
print(f"  Devices    : {n_dev} ({n_dev//2} healthy, {n_dev//2} under attack)")
print(f"  Timesteps  : {df_sim.groupby('device_id').size().iloc[0]} per device  |  {len(df_sim):,} total rows")
print(f"  Features   : {df_sim.shape[1] - 2}  (CPU, memory, network, vitals, etc.)")
print(f"  Attack starts at timestep 200 out of 500")
print()
print("=== TON-IoT Network Dataset (test split) ===")
print(f"  Rows     : {X_ton.shape[0]:,}")
print(f"  Features : {X_ton.shape[1]}  (network connection attributes)")
print(f"  Attack % : {y_ton.mean()*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df_sim["label"].value_counts().sort_index()
axes[0].bar(["Healthy", "Attack"], counts.values, color=["#2ecc71", "#e74c3c"], edgecolor="black", width=0.5)
axes[0].set_title("Simulated ICU — How Much of the Data is Attack?", fontweight="bold")
axes[0].set_ylabel("Timestep Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 150, f"{v:,}", ha="center", fontsize=10)

device_type = df_sim["device_id"].str.extract(r"^([a-z_]+?)(?:_attack|_benign)")[0]
df_sim["device_type"] = device_type
dev_counts = df_sim.groupby(["device_type", "status"]).size().unstack(fill_value=0)
dev_counts.plot(kind="bar", ax=axes[1], color=["#e74c3c", "#2ecc71"], edgecolor="black")
axes[1].set_title("Simulated ICU — Device Types", fontweight="bold")
axes[1].set_ylabel("Timestep Count")
axes[1].tick_params(axis="x", rotation=20)
axes[1].legend(title="Status")

plt.tight_layout()
plt.show()

### Why these two datasets?

| | Simulated ICU | TON-IoT |
|---|---|---|
| **What it contains** | Device sensor readings (CPU, memory, network, vitals) | Real network traffic logs |
| **Attack timing** | Known — starts at timestep 200 | Unknown — attacks mixed throughout |
| **Device types** | Ventilators, infusion pumps, monitors, gateways | Network connections across attack categories |
| **What it tests** | Early detection — we know exactly when the attack starts | Generalisation — does the model work on real-world traffic? |

Having both datasets matters. The simulated ICU lets us measure precisely *how early* the model detects an attack. TON-IoT checks whether the pipeline holds up on messier, real-world data with a wider variety of attack types.

### CICIoMT2024 — Third Dataset (EDA Complete)

We are adding a third dataset covering WiFi/MQTT and Bluetooth medical device traffic. Exploratory analysis is done; the preprocessing pipeline is currently being built.

In [ ]:
cic_figs = [
    ("results/figures/cic_label_distribution.png",   "What types of traffic are in the dataset?"),
    ("results/figures/cic_protocol_composition.png", "WiFi/MQTT vs Bluetooth breakdown"),
    ("results/figures/cic_anomaly_gap.png",          "How separable are attack and normal traffic?"),
    ("results/figures/cic_feature_importance.png",   "Which features are most useful for detection?"),
]
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (path, title) in zip(axes.flatten(), cic_figs):
    if os.path.exists(path):
        ax.imshow(mpimg.imread(path))
    else:
        ax.text(0.5, 0.5, "Figure not found", ha="center", va="center", transform=ax.transAxes)
    ax.set_title(title, fontweight="bold")
    ax.axis("off")
plt.suptitle("CICIoMT2024 — Exploratory Analysis", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

**What the CIC analysis tells us:**
- The **anomaly gap** chart shows that attack and normal traffic look quite different from each other in this dataset — a promising sign that our pipeline should detect attacks here too.
- **Feature importance** shows a clear subset of network features that do most of the heavy lifting for classification, which will help us build a focused preprocessing pipeline.
- The dataset covers two communication protocols (WiFi/MQTT and Bluetooth), which adds breadth since many medical devices use both.

---
## 3. How the Pipeline Works

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

X_sim_test = np.load(os.path.join(cfg.sim_splits_path, "X_test.npy"))
ae_sim  = Autoencoder(X_sim_test.shape[1], cfg.ae_hidden_dim1, cfg.ae_hidden_dim2, cfg.latent_dim)
ae_ton  = Autoencoder(X_ton.shape[1],      cfg.ae_hidden_dim1, cfg.ae_hidden_dim2, cfg.latent_dim)
mamba   = MambaClassifier(cfg.latent_dim + 1, cfg.d_model, cfg.num_layers, cfg.dropout)
lstm    = LSTMClassifier( cfg.latent_dim + 1, cfg.d_model, cfg.num_layers, cfg.dropout)

print("STAGE 1 — Autoencoder")
print(f"  SIM  : {X_sim_test.shape[1]} sensor features  →  32 compressed features  →  reconstruct")
print(f"  TON  : {X_ton.shape[1]} network features  →  32 compressed features  →  reconstruct")
print(f"  Parameters: SIM={count_params(ae_sim):,}  |  TON={count_params(ae_ton):,}")
print()
print("  Output passed to Stage 2: 32 compressed features + 1 reconstruction error score = 33 values per timestep")
print()
print("STAGE 2 — Sequence Classifier  (window of 20 timesteps at a time)")
print(f"  Mamba  [{count_params(mamba):,} parameters]")
print(f"  LSTM   [{count_params(lstm):,} parameters]  — used as a comparison baseline")
print()
print(f"Key settings:  window size={cfg.seq_len} timesteps,  compressed size={cfg.latent_dim},  decision threshold={cfg.threshold}")

### Stage 1 — The Autoencoder

The autoencoder is trained only on data from healthy devices. It learns to compress a timestep of sensor readings down to 32 numbers (the "latent" representation), then reconstruct the original readings from those 32 numbers.

Once trained, when we feed it a timestep from a device under attack, it struggles to reconstruct accurately — because it was never trained on attack patterns. That reconstruction error becomes our anomaly score.

```
Input features  →  [compress to 32]  →  [reconstruct]  →  Reconstruction error
```

### Stage 2 — The Mamba Classifier

Instead of treating each timestep independently, we pass a window of the last 20 timesteps through the Mamba model. It reads the sequence from start to finish, learning whether the *pattern across time* looks like an attack building up.

```
20 timesteps × 33 features  →  Mamba reads sequence  →  Attack probability (0–1)
```

**Why Mamba and not something simpler?**
A standard single-timestep classifier can only react once a device looks obviously compromised. Mamba can pick up on the gradual build-up — several slightly unusual readings in a row — before any single timestep would trigger an alarm on its own.

We also trained an LSTM (a well-established sequence model) as a comparison, and a simple Logistic Regression (single timestep only) as a baseline.

---
## 4. Training

In [ ]:
loss_path = "results/figures/loss_curves.png"
if os.path.exists(loss_path):
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.imshow(mpimg.imread(loss_path))
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    specs = [
        ("Autoencoder — Reconstruction Loss", ["ae_sim_losses.npy",    "ae_ton_losses.npy"]),
        ("Mamba — Classification Loss",       ["mamba_sim_losses.npy", "mamba_ton_losses.npy"]),
        ("LSTM — Classification Loss",        ["lstm_sim_losses.npy",  "lstm_ton_losses.npy"]),
    ]
    for ax, (title, files) in zip(axes, specs):
        for fname, label, color in zip(files, ["SIM", "TON-IoT"], ["#2563EB", "#DC2626"]):
            p = os.path.join(cfg.loss_dir, fname)
            if os.path.exists(p):
                l = np.load(p)
                ax.plot(range(1, len(l)+1), l, label=label, color=color, linewidth=2)
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Training Epoch"); ax.set_ylabel("Loss")
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.suptitle("How Loss Changed During Training", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()

# Show how much each model improved
print(f"  {'Model':<14}  {'Start Loss':>10}  {'End Loss':>9}  {'Improvement':>12}")
print(f"  {'-'*14}  {'-'*10}  {'-'*9}  {'-'*12}")
for label, fname in [
    ("AE (SIM)",    "ae_sim_losses.npy"),
    ("AE (TON)",    "ae_ton_losses.npy"),
    ("Mamba (SIM)", "mamba_sim_losses.npy"),
    ("Mamba (TON)", "mamba_ton_losses.npy"),
    ("LSTM (SIM)",  "lstm_sim_losses.npy"),
    ("LSTM (TON)",  "lstm_ton_losses.npy"),
]:
    p = os.path.join(cfg.loss_dir, fname)
    if os.path.exists(p):
        l = np.load(p)
        print(f"  {label:<14}  {l[0]:>10.4f}  {l[-1]:>9.4f}  {(1-l[-1]/l[0])*100:>11.1f}%")

### What these curves show

Each curve tracks how much error the model made during training, epoch by epoch. A curve that drops smoothly and levels off means the model learned well — it found real structure in the data rather than just memorising it.

**The autoencoder** converging cleanly is important: it means the 32-number compression genuinely captures the patterns of healthy device behaviour. If it didn't learn properly, the reconstruction error would be noisy and useless as an anomaly signal.

**The classifiers** typically converge within 5–7 epochs. The simulated ICU data trains faster because it's clean and consistent. The TON-IoT data is more varied (real-world network traffic from multiple attack types), so training takes slightly longer — but both reach stable, low loss values.

---
## 5. Results

We run inference on the held-out test set — data the models have never seen during training.

In [ ]:
def load_ae(path, input_dim, cfg, device):
    ae = Autoencoder(input_dim, cfg.ae_hidden_dim1, cfg.ae_hidden_dim2, cfg.latent_dim).to(device)
    ae.load_state_dict(torch.load(path, map_location=device))
    ae.eval()
    return ae

def encode(ae, X, device, batch_size=256):
    # Pass data through the autoencoder to get compressed features + reconstruction error
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            batch = torch.tensor(X[i:i+batch_size], dtype=torch.float32).to(device)
            z, x_hat = ae(batch)
            err = torch.mean((batch - x_hat)**2, dim=1, keepdim=True)
            out.append(torch.cat([z, err], dim=1).cpu().numpy())
    return np.vstack(out)

def run_classifier(model, dataset, cfg, device):
    # Run a sequence classifier and collect predicted probabilities
    loader = DataLoader(dataset, batch_size=cfg.clf_batch_size, shuffle=False)
    probs, labels = [], []
    model.eval()
    with torch.no_grad():
        for X_b, y_b in loader:
            probs.append(torch.sigmoid(model(X_b.to(device))).cpu().numpy())
            labels.append(y_b.numpy())
    return np.concatenate(probs), np.concatenate(labels)

datasets_cfg = {
    "Simulated ICU": dict(
        split_dir=cfg.sim_splits_path, ae_path=cfg.sim_autoencoder_model_path,
        clf_path=cfg.sim_classifier_model_path, lstm_path=cfg.sim_lstm_model_path, label_mode="any"),
    "TON-IoT": dict(
        split_dir=cfg.ton_splits_path, ae_path=cfg.ton_autoencoder_model_path,
        clf_path=cfg.ton_classifier_model_path, lstm_path=cfg.ton_lstm_model_path, label_mode="last"),
}

results = {}
print("Loading models and running on test data...")
for ds_name, ds in datasets_cfg.items():
    X_test  = np.load(os.path.join(ds["split_dir"], "X_test.npy"))
    y_test  = np.load(os.path.join(ds["split_dir"], "y_test.npy"))
    g_test  = np.load(os.path.join(ds["split_dir"], "group_ids_test.npy"), allow_pickle=True)
    ae      = load_ae(ds["ae_path"], X_test.shape[1], cfg, device)
    Z_test  = encode(ae, X_test, device)
    test_ds = ArraySequenceDataset(Z_test, y_test, g_test, cfg.seq_len, ds["label_mode"])
    mamba_m = MambaClassifier(cfg.latent_dim+1, cfg.d_model, cfg.num_layers, cfg.dropout).to(device)
    mamba_m.load_state_dict(torch.load(ds["clf_path"],  map_location=device))
    lstm_m  = LSTMClassifier( cfg.latent_dim+1, cfg.d_model, cfg.num_layers, cfg.dropout).to(device)
    lstm_m.load_state_dict( torch.load(ds["lstm_path"], map_location=device))
    mp, y_true = run_classifier(mamba_m, test_ds, cfg, device)
    lp, _      = run_classifier(lstm_m,  test_ds, cfg, device)
    results[ds_name] = {
        "Mamba": (compute_metrics(y_true.astype(int), (mp>=cfg.threshold).astype(int), mp), mp, y_true),
        "LSTM":  (compute_metrics(y_true.astype(int), (lp>=cfg.threshold).astype(int), lp), lp, y_true),
    }
    print(f"  {ds_name} — done")
print("Complete.")

In [ ]:
# Performance table
metric_keys   = ["accuracy", "precision", "recall", "f1", "auc_roc"]
metric_labels = ["Accuracy", "Precision", "Recall", "F1 Score", "AUC-ROC"]
rows = []
for ds_name, models in results.items():
    for model_name, (m, _, __) in models.items():
        row = {"Dataset": ds_name, "Model": model_name}
        row.update({lbl: f"{m[k]:.4f}" for k, lbl in zip(metric_keys, metric_labels)})
        rows.append(row)
display(pd.DataFrame(rows).set_index(["Dataset", "Model"]))

In [ ]:
# ROC curves and confusion matrices
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.4, wspace=0.35)
colors = {"Mamba": "#2563EB", "LSTM": "#DC2626"}

for col, (ds_name, models) in enumerate(results.items()):
    ax_roc = fig.add_subplot(gs[0, col*2:col*2+2])
    for model_name, (m, prob, y_true) in models.items():
        fpr, tpr, _ = roc_curve(y_true.astype(int), prob)
        ax_roc.plot(fpr, tpr, label=f"{model_name} (AUC={m['auc_roc']:.3f})",
                    color=colors[model_name], linewidth=2)
    ax_roc.plot([0,1],[0,1],"k--",linewidth=1,alpha=0.4,label="Random guess")
    ax_roc.set_title(f"ROC Curve — {ds_name}", fontweight="bold")
    ax_roc.set_xlabel("False Alarm Rate"); ax_roc.set_ylabel("Detection Rate")
    ax_roc.legend(); ax_roc.grid(True, alpha=0.3)

    m, prob, y_true = models["Mamba"]
    cm = m["confusion_matrix"]
    ax_cm = fig.add_subplot(gs[1, col*2:col*2+2])
    im = ax_cm.imshow(cm, cmap="Blues")
    ax_cm.set_title(f"Confusion Matrix — {ds_name} (Mamba)", fontweight="bold")
    ax_cm.set_xlabel("Predicted Label"); ax_cm.set_ylabel("True Label")
    ax_cm.set_xticks([0,1]); ax_cm.set_yticks([0,1])
    ax_cm.set_xticklabels(["Healthy","Attack"]); ax_cm.set_yticklabels(["Healthy","Attack"])
    for i in range(2):
        for j in range(2):
            ax_cm.text(j, i, f"{cm[i,j]:,}", ha="center", va="center",
                       color="white" if cm[i,j] > cm.max()/2 else "black", fontsize=12)
    plt.colorbar(im, ax=ax_cm, fraction=0.046)

plt.suptitle("Detection Performance", fontsize=14, fontweight="bold")
plt.show()

In [ ]:
# Best model summary with plain-language output
lines = []
for ds_name, models in results.items():
    best_name = max(models.items(), key=lambda x: x[1][0]["f1"])[0]
    m, _, __ = models[best_name]
    cm = m["confusion_matrix"]
    tn, fp, fn, tp = cm.ravel()
    total = tn + fp + fn + tp
    lines += [
        f"### {ds_name} — Best model: {best_name}",
        "",
        "| Metric | Score | What it means |",
        "|--------|-------|----------------|",
        f"| Accuracy  | **{m['accuracy']:.4f}** | Overall correct predictions |",
        f"| Recall    | **{m['recall']:.4f}**   | Share of actual attacks caught |",
        f"| Precision | **{m['precision']:.4f}** | Share of alarms that are real attacks |",
        f"| F1 Score  | **{m['f1']:.4f}**       | Balance of recall and precision |",
        f"| AUC-ROC   | **{m['auc_roc']:.4f}**  | Overall ability to separate attack from normal |",
        "",
        f"Out of **{total:,}** test windows: {tp:,} attacks correctly flagged, "
        f"{fn:,} attacks missed, {fp:,} false alarms, {tn:,} healthy windows correctly cleared.",
        "",
    ]
display(Markdown("\n".join(lines)))

### Reading the results

**Recall** is the most important number for our use case — it tells us what fraction of actual attacks the model caught. A missed attack means a device keeps encrypting undetected, so we want this as close to 1.0 as possible.

**Precision** tells us how often an alarm is a real attack vs a false alarm. False alarms matter because if they happen too often, clinical staff start ignoring them. We want both recall and precision to be high.

**AUC-ROC** summarises performance across all possible alarm thresholds. A score of 1.0 is perfect; 0.5 is no better than random. Our scores close to 1.0 indicate the model's confidence scores cleanly separate attacks from normal traffic — meaning we have flexibility to tune the alarm threshold for different deployment settings.

**Why TON-IoT scores slightly lower than SIM:** The simulated ICU has a controlled, consistent attack pattern. TON-IoT is messier — real network traffic from multiple attack types mixed together. The difference reflects genuine difficulty, not a bug.

---
## 6. Ablation Study — Does Each Part Actually Help?

Before committing to the full pipeline, we tested simpler versions to check that each component earns its place.

| Variant | What it is |
|---|---|
| **AE Only** | Just the autoencoder's reconstruction error — no classifier |
| **AE + Logistic Regression** | A simple single-timestep classifier on the compressed features |
| **AE + LSTM** | A sequence classifier (reads 20 timesteps, standard approach) |
| **AE + Mamba** | Our full pipeline |

If each step genuinely adds value, we should see performance improve at each row.

In [ ]:
def abl_metrics(y_true, y_pred, y_prob):
    return {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
        "F1":        f1_score(y_true, y_pred, zero_division=0),
        "AUC-ROC":   roc_auc_score(y_true, y_prob),
    }

all_ablation = {}
print("Running ablation study...")
for ds_name, ds in datasets_cfg.items():
    X_train = np.load(os.path.join(ds["split_dir"], "X_train.npy"))
    X_test  = np.load(os.path.join(ds["split_dir"], "X_test.npy"))
    y_train = np.load(os.path.join(ds["split_dir"], "y_train.npy"))
    y_test  = np.load(os.path.join(ds["split_dir"], "y_test.npy"))
    g_train = np.load(os.path.join(ds["split_dir"], "group_ids_train.npy"), allow_pickle=True)
    g_test  = np.load(os.path.join(ds["split_dir"], "group_ids_test.npy"),  allow_pickle=True)

    ae   = load_ae(ds["ae_path"], X_test.shape[1], cfg, device)
    Z_tr = encode(ae, X_train, device)
    Z_te = encode(ae, X_test,  device)

    train_ds = ArraySequenceDataset(Z_tr, y_train, g_train, cfg.seq_len, ds["label_mode"])
    test_ds  = ArraySequenceDataset(Z_te, y_test,  g_test,  cfg.seq_len, ds["label_mode"])
    y_seq    = test_ds.labels.numpy().astype(int)

    # AE Only — flag anything with reconstruction error above a threshold
    benign_err = Z_tr[:, -1][y_train == 0]
    thr        = benign_err.mean() + 2 * benign_err.std()
    ae_scores  = test_ds.samples[:, -1, -1].numpy()

    # AE + Logistic Regression — single-timestep classifier
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(train_ds.samples[:, -1, :].numpy(), train_ds.labels.numpy().astype(int))
    lr_prob = lr.predict_proba(test_ds.samples[:, -1, :].numpy())[:, 1]

    # AE + LSTM and AE + Mamba — load trained sequence classifiers
    mamba_m = MambaClassifier(cfg.latent_dim+1, cfg.d_model, cfg.num_layers, cfg.dropout).to(device)
    mamba_m.load_state_dict(torch.load(ds["clf_path"],  map_location=device))
    lstm_m  = LSTMClassifier( cfg.latent_dim+1, cfg.d_model, cfg.num_layers, cfg.dropout).to(device)
    lstm_m.load_state_dict( torch.load(ds["lstm_path"], map_location=device))

    mp, _ = run_classifier(mamba_m, test_ds, cfg, device)
    lp, _ = run_classifier(lstm_m,  test_ds, cfg, device)

    all_ablation[ds_name] = {
        "AE Only":    abl_metrics(y_seq, (ae_scores>=thr).astype(int), ae_scores),
        "AE + LR":    abl_metrics(y_seq, (lr_prob>=cfg.threshold).astype(int), lr_prob),
        "AE + LSTM":  abl_metrics(y_seq, (lp>=cfg.threshold).astype(int), lp),
        "AE + Mamba": abl_metrics(y_seq, (mp>=cfg.threshold).astype(int), mp),
    }
    print(f"  {ds_name} done.")
print("Done.")

In [ ]:
model_names  = ["AE Only", "AE + LR", "AE + LSTM", "AE + Mamba"]
bar_colors   = ["#059669", "#DC2626", "#F59E0B", "#2563EB"]
metric_names = ["Accuracy", "Precision", "Recall", "F1", "AUC-ROC"]
x, w = np.arange(len(metric_names)), 0.20

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Does Each Component Add Value?", fontsize=14, fontweight="bold")
for ax, (ds_name, ds_res) in zip(axes, all_ablation.items()):
    for mn, color, offset in zip(model_names, bar_colors, [-1.5, -0.5, 0.5, 1.5]):
        vals = [ds_res[mn][m] for m in metric_names]
        bars = ax.bar(x + offset*w, vals, w, label=mn, color=color, alpha=0.85)
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x()+bar.get_width()/2, h+0.002, f"{h:.3f}",
                    ha="center", va="bottom", fontsize=7, rotation=90)
    ax.set_xticks(x); ax.set_xticklabels(metric_names, fontsize=10)
    ax.set_ylim([0.85, 1.03]); ax.set_ylabel("Score")
    ax.set_title(ds_name, fontsize=12)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

# F1 gain table
lines = []
for ds_name, ds_res in all_ablation.items():
    lines += [
        f"### {ds_name} — F1 Score at Each Stage",
        "| Model | F1 | Improvement |",
        "|-------|----|-------------|",
    ]
    prev = None
    for mn in model_names:
        f1   = ds_res[mn]["F1"]
        gain = f"+{f1-prev:.4f}" if prev is not None else "baseline"
        lines.append(f"| {mn} | {f1:.4f} | {gain} |")
        prev = f1
    lines.append("")
display(Markdown("\n".join(lines)))

### What this shows

**AE Only → AE + LR:** Adding even a simple classifier on top of the autoencoder features improves things. This tells us the compressed features contain more useful information than just the reconstruction error score alone.

**AE + LR → AE + LSTM:** Switching from a single-timestep classifier to one that reads a sequence of 20 timesteps gives another clear improvement. This is the most important result in the ablation — it confirms that **the pattern across time matters**, not just how any one moment looks.

**AE + LSTM → AE + Mamba:** Mamba outperforms the LSTM. The selective memory mechanism — learning which parts of the sequence to pay attention to — gives it an edge for detecting the gradual build-up of an attack.

---
## 7. How Early Can We Catch It?

Knowing the model is accurate is good. Knowing *how quickly* it raises an alarm is what matters clinically.

In the simulated ICU dataset, every attacked device starts showing ransomware behaviour at **timestep 200** (out of 500). We measure how many timesteps after that each model first fires an alarm — the detection lag.

In [ ]:
ATTACK_START = 200

def sliding_windows(Z, seq_len):
    # Split a device's full timeseries into overlapping 20-timestep windows
    N, F = Z.shape
    if N < seq_len: return np.empty((0, seq_len, F), dtype=Z.dtype)
    idx = np.arange(seq_len)[None,:] + np.arange(N - seq_len + 1)[:,None]
    return Z[idx]

def first_detection(preds, seq_len):
    # Find the timestep of the model's first alarm
    hits = np.where(preds == 1)[0]
    return None if len(hits) == 0 else hits[0] + seq_len - 1

# Load SIM data and models
X_train = np.load(os.path.join(cfg.sim_splits_path, "X_train.npy"))
X_test  = np.load(os.path.join(cfg.sim_splits_path, "X_test.npy"))
y_train = np.load(os.path.join(cfg.sim_splits_path, "y_train.npy"))
g_train = np.load(os.path.join(cfg.sim_splits_path, "group_ids_train.npy"), allow_pickle=True)
g_test  = np.load(os.path.join(cfg.sim_splits_path, "group_ids_test.npy"),  allow_pickle=True)

ae   = load_ae(cfg.sim_autoencoder_model_path, X_test.shape[1], cfg, device)
Z_tr = encode(ae, X_train, device)
Z_te = encode(ae, X_test,  device)

mamba_m = MambaClassifier(cfg.latent_dim+1, cfg.d_model, cfg.num_layers, cfg.dropout).to(device)
mamba_m.load_state_dict(torch.load(cfg.sim_classifier_model_path, map_location=device)); mamba_m.eval()
lstm_m  = LSTMClassifier( cfg.latent_dim+1, cfg.d_model, cfg.num_layers, cfg.dropout).to(device)
lstm_m.load_state_dict( torch.load(cfg.sim_lstm_model_path,       map_location=device)); lstm_m.eval()

# Train LR on training data
u_tr, inv_tr = np.unique(g_train, return_inverse=True)
tr_last, tr_labs = [], []
for gi in range(len(u_tr)):
    rows = np.where(inv_tr == gi)[0]
    if len(rows) < cfg.seq_len: continue
    wins = sliding_windows(Z_tr[rows], cfg.seq_len)
    yr   = y_train[rows]
    labs = [int(np.any(yr[j:j+cfg.seq_len])) for j in range(len(rows)-cfg.seq_len+1)]
    tr_last.append(wins[:, -1, :]); tr_labs.append(labs)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(np.vstack(tr_last), np.concatenate(tr_labs))

# Run per-device detection on test set
u_te, inv_te = np.unique(g_test, return_inverse=True)
mamba_lags, lstm_lags, lr_lags = [], [], []
n_attacked = 0

for gi in range(len(u_te)):
    rows   = np.where(inv_te == gi)[0]
    dev_id = u_te[gi]
    if "attack" not in str(dev_id): continue
    n_attacked += 1
    Z_dev = Z_te[rows]
    if len(Z_dev) < cfg.seq_len: continue
    wins = sliding_windows(Z_dev, cfg.seq_len)
    with torch.no_grad():
        X_t = torch.tensor(wins, dtype=torch.float32).to(device)
        mp  = (torch.sigmoid(mamba_m(X_t)) >= cfg.threshold).cpu().numpy().astype(int)
        lp  = (torch.sigmoid(lstm_m(X_t))  >= cfg.threshold).cpu().numpy().astype(int)
    lrp = lr.predict(wins[:, -1, :])
    for preds, lags in [(mp, mamba_lags), (lp, lstm_lags), (lrp, lr_lags)]:
        t = first_detection(preds, cfg.seq_len)
        if t is not None and t - ATTACK_START >= 0:
            lags.append(t - ATTACK_START)

print(f"Attacked devices in test set: {n_attacked}")
for label, lags in [("Mamba", mamba_lags), ("LSTM", lstm_lags), ("LR", lr_lags)]:
    print(f"  {label:<6} detected {len(lags)}/{n_attacked}")

In [ ]:
steps      = list(range(0, 101, 5))
model_lags = [("Mamba", mamba_lags, "#2563EB"), ("LSTM", lstm_lags, "#DC2626"), ("LR (baseline)", lr_lags, "#059669")]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cumulative detection rate — how quickly does each model catch attacks?
for label, lags, color in model_lags:
    cum = [sum(1 for l in lags if l <= s)/n_attacked for s in steps]
    axes[0].plot(steps, cum, label=f"{label} ({len(lags)}/{n_attacked})", color=color, linewidth=2)
axes[0].axvline(0, color="black", linestyle="--", linewidth=1, alpha=0.5)
axes[0].set_title("How Quickly Is Each Attack Caught?", fontweight="bold")
axes[0].set_xlabel("Timesteps After Attack Starts")
axes[0].set_ylabel("Fraction of Attacked Devices Detected")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Distribution of detection lags
lag_data  = [l for _, l, _ in model_lags if l]
lag_names = [n for n, l, _ in model_lags if l]
lag_cols  = [c for _, l, c in model_lags if l]
bp = axes[1].boxplot(lag_data, patch_artist=True, labels=lag_names)
for patch, color in zip(bp["boxes"], lag_cols):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1].set_title("Detection Lag — How Long After Attack Start?", fontweight="bold")
axes[1].set_ylabel("Timesteps After Attack Starts")
axes[1].grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

# Summary table
lines = [
    "### Detection Lag Summary\n",
    "| Model | Devices Detected | Median Lag | Fastest Detection | Slowest Detection |",
    "|-------|-----------------|------------|-------------------|-------------------|",
]
for label, lags, _ in model_lags:
    if lags:
        lines.append(
            f"| {label} | {len(lags)} / {n_attacked} | {np.median(lags):.0f} timesteps | "
            f"{int(np.min(lags))} timesteps | {int(np.max(lags))} timesteps |"
        )
    else:
        lines.append(f"| {label} | 0 / {n_attacked} | — | — | — |")
display(Markdown("\n".join(lines)))

### What this means in practice

The **cumulative detection rate curve** shows what percentage of attacked devices each model has flagged by a given number of timesteps after the attack starts. A curve that rises steeply and early is what we want.

**Median lag** is the most practically useful number — it tells us that in a typical case, the model raises an alarm this many timesteps after the attack begins. In a system with 1-second sampling intervals, a median lag of 20 means the typical alarm arrives 20 seconds into the attack.

**Why Mamba and LSTM detect earlier than Logistic Regression:** LR looks at one timestep at a time. It can only fire an alarm once the attack is obvious in the current moment's data. Mamba and LSTM can detect the *build-up* — a sequence of slightly unusual readings — before any single moment would be clearly anomalous on its own. That is the core advantage of temporal modelling for this problem.

---
## 8. Summary & What's Next

### What We've Built

A two-stage pipeline that:
1. Learns normal device behaviour from healthy data alone (Autoencoder)
2. Classifies 20-timestep windows as attack or benign using a Mamba sequence model
3. Has been trained, validated, and ablation-tested on two datasets
4. Detects attacks early — before a single-timestep classifier would

### Current Status

| Component | Status |
|-----------|--------|
| Simulated ICU — training & evaluation | Done |
| TON-IoT — training & evaluation | Done |
| Ablation study | Done |
| Early detection analysis | Done |
| Statistical significance testing (McNemar's) | Done |
| CICIoMT2024 — exploratory analysis | Done |
| CICIoMT2024 — preprocessing & training | In progress |
| Cross-dataset generalisation testing | In progress |

### Next Steps

1. **Finish CICIoMT2024 pipeline** — preprocess WiFi/MQTT and Bluetooth data, train and evaluate on it
2. **Cross-dataset testing** — train on one dataset, evaluate on another, to see how well the model generalises
3. **Sequence length study** — test whether windows of 10, 15, 20, 30, or 50 timesteps give better early detection
